# Introduction

In this notebook, we first introduce the high-level architecture of NVIDIA FLARE. Then, we walkthrough core user APIs of NVIDIA FLARE which allow scientists and developers to quickly build fundamental components of federated learning and adapt any centralized computation routine to federated paradigm. Finally, we finish this notebook by putting all elements together with an example of federated numerical computation using `numpy`. The example uses FL Simulator, a handy runtime tool that allows researchers to quickly test-run a federated job on a local development PC.


# NVIDIA FLARE Architecture

The diagram below summarizes the high-level architecture of NVIDIA FLARE.

<img src="images/nvflare-arch.png" alt="NVFLARE Arch" width=400/>

Now let's look at this diagram in more details. In NVIDIA FLARE, a federated workflow is centered around the interaction between server-side "Controller" and client-side "Executors", through the concept of "Tasks", which can be local training, local validation, or any other general routines that happen on the client side. The interaction between server and client is defined and conceptualized as a "Federated Job". NVIDIA FLARE provides "Runtime" to run different jobs. Here are details on these fundamental components:
- **Server-side Controller**: a Controller defines the server-side aggregation workflow and coordinates with client-side Executors through task assignments. NVIDIA FLARE provides server-side Controller classes with extensible APIs which allow developers to re-use classic federated workflows (scatter-and-gather, cyclic etc.) and classic optimization algorithms (FedAvg, FedOpt, etc.) or to efficiently implement customized workflows.
- **Client-side Executor**: an Executor implements details and logics on how to perform the task received from the server-side Controller, whether the task is local training, or a general compute routine. In NVIDIA FLARE, client-side Executors can be created using intuitive Python APIs, allowing researchers to easily convert an existing centralized computation / training codes to a federated / distributed paradigm.
- **Federated Job**: NVIDIA FLARE provides `FedJob` class, an abstraction of server-client interaction, which allow users to set up and configure a federated workflow. A `FedJob` can be exported and run by NVIDIA FLARE runtime.
- **Runtime**: NVIDIA FLARE provides different runtime backends to run federated jobs, either in a simulated environment to facilitate research and fast debugging & prototyping, or in a real-world scenario with capabilities to monitor and manage multiple jobs. We will focus the FL Simulator in this notebook, which allows researchers to test-run federated jobs in a simulated environment on a local development PC, before real-world deployment.

Notice also the possibility of adding filters to task data and / or results, at any moment of the Controller & Executor interaction. This filtering mechanism provides a flexible way to add security & privacy filters, for example homomorphic encryption and differential privacy filters.


# Core User APIs

NVIDIA FLARE offers a rich set of APIs. In this notebook, we look at the core user APIs that allow you to adapt a centralized compute routine to federated paradigm in 5 minutes. These APIs can be summarized in 3 categories:
- **Server-side APIs**: these are essentially APIs for implementing server-side Controllers. In NVIDIA FLARE, implementing server-side Controllers are made easy with the [ModelController](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/model_controller.py#L23) class. This is a basic class allowing you to either easily re-use classic federated workflow, or customize your own workflow.
- **Client-side APIs**: these are essentially APIs for implementing client-side Executors. NVIDIA FLARE provides [Client APIs](https://nvflare.readthedocs.io/en/main/programming_guide/execution_api_type/client_api.html#client-api), a suite of carefully designed APIs that allow user to easily adapt any centralized compute to federated compute.
- **Job APIs**: these are essentially APIs for creating a Federated Job, to be run using NVIDIA FLARE runtime. More specifically, we will look at the [FedJob](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/job_config/api.py#L147) class.

In the following section, we will demonstrate the use of these core APIs with a toy computation example using `numpy`.


# Example: Federated `numpy`

This **Hello FedAvg NumPy** example is taken from NVIDIA FLARE's [example repository](https://github.com/NVIDIA/NVFlare/tree/109da964126c015d248718dbff7865206aa2ad6f/examples/hello-world/hello-fedavg-numpy). In this toy example, we will implement a simple distributed workflow, where a number of clients will perform simple modifications of a fixed `numpy` array, independently for a certain number of iterations, and a server will aggregate the arrays modified by each client. More specifically, for each round: 
- On the client-side, each client will perform simple arithmetics on a fixed `numpy` array separately in parallel
- On the server-side, the server will simply get the average value of the arrays modified by the clients

This whole process will be repeated a couple of times.

We will show how you can easily implement this example using NVIDIA FLARE's **Server, Client and Job APIs.**


### Setup

Let's first copy the example to our notebook workspace

In [1]:
!if [ ! -d examples ]; then mkdir examples; fi
!if [ ! -d examples/hello-fedavg-numpy ]; then \
    cp -r ../NVFlare/examples/hello-world/hello-fedavg-numpy examples/hello-fedavg-numpy; fi
!tree examples/hello-fedavg-numpy

examples/hello-fedavg-numpy
├── README.md
├── fedavg_script_runner_hello-numpy.py
├── hello-fedavg-numpy_flare_api.ipynb
├── hello-fedavg-numpy_getting_started.ipynb
├── requirements.txt
└── src
    └── hello-numpy_fl.py

2 directories, 6 files


You can see that we have many files in this example. In this notebook, we will ignore the `.ipynb` files in the example, and focus solely on the two Python source code files: 
- [examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py](examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py)
- [examples/hello-fedavg-numpy/src/hello-numpy_fl.py](examples/hello-fedavg-numpy/src/hello-numpy_fl.py).

To give you the best understanding of FLARE's APIs, we will walkthrough the two Python files in details, in the following order:
- First, we will look at the client-side implementation. This is essentially the content of file [examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py](examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py). The client-side implementation, as we've explained above, defines the local local computation routine of each site.
- Then, we will look at the server-side implementation. This is part of the file [examples/hello-fedavg-numpy/src/hello-numpy_fl.py](examples/hello-fedavg-numpy/src/hello-numpy_fl.py). The server-side implementation, as we've explained above, defines the server side aggregation workflow.
- Finally, we will look at how to connect the server- and client-side implementations together, by wrapping them into a Federated Job. This is the rest of the file [examples/hello-fedavg-numpy/src/hello-numpy_fl.py](examples/hello-fedavg-numpy/src/hello-numpy_fl.py).


### Client-Side Implementation

Let's start with client-side implementation, which is in file: [examples/hello-fedavg-numpy/src/hello-numpy_fl.py](examples/hello-fedavg-numpy/src/hello-numpy_fl.py). 

As already mentioned above in FLARE's architecture, the client-side implements Executors which handles the computation routine / training logic that happens locally on each client site. In a traditional centralized computation paradigm, this is where the core computation / training part happens. 

Starting from NVIDIA FLARE version 2.4, we've introduced [**FLARE client APIs**](https://nvflare.readthedocs.io/en/main/programming_guide/execution_api_type/client_api.html), which allow you to convert a traditional centralized computation code to federated, simply by adding a couple of lines of code. Then, using some convenient APIs, you can create client-side Executors from the modified computation code. **These new client APIs alleviate the need of manually writing boilerplate code for creating and configuring Executors, making it extremely easy and seamingless for researchers to adapt existing centralized computation to a federated paradigm**.

To illustrate this, let's analyze the client code in file: [examples/hello-fedavg-numpy/src/hello-numpy_fl.py](examples/hello-fedavg-numpy/src/hello-numpy_fl.py). Let's first remove all NVIDIA FLARE related code. After doing this, we have the following: 
```python
import copy
import numpy as np

def train(input_arr):
    output_arr = copy.deepcopy(input_arr)
    # mock training with plus 1
    return output_arr + 1

def evaluate(input_arr):
    # mock evaluation metrics
    return np.mean(input_arr)

def main():
    # This is our "mock" model that we aim to "train"
    input_model = ...

    # Get parameters from "mock" input_model: if empty, use default value.
    if input_model.params == {}:
        params = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=np.float32)
    else:
        params = np.array(input_model.params["numpy_key"], dtype=np.float32)
        
    # mock training
    new_params = train(params)
    
    # mock evaluation
    metrics = evaluate(params)

if __name__ == "__main__":
    main()
```

You can see that this is nothing but a simple mock "training" & "evaluation" routine using `numpy`:
- Looking at the function `main()`, which describes the client-side compute flow: the model we want to "train" in this example is an object named `input_model`. The way how we get the `input_model` object is not important for the moment. Our goal here is to "train" its parameters, represented by the `params["numpy_key"]` attribute, which is an `numpy` array. We first get a copy of the parameters `input_model.params["numpy_key"]` and assign to an array named `params`. If `input_model.params["numpy_key"]` is empty (for example, during the initialization round), we use the default value `[[1, 2, 3], [4, 5, 6], [7, 8, 9]]`.
- The `train()` function performs a mock "training" task: adding value `1` to each element of `params` array.
- The `evaluate()` function perform a mock "evaluation", by simply computing the mean value of the array `params`.

**Now let's make this simple computation federated: we will show you step-by-step how easy it is to do that by adding NVIDIA FLARE's client APIs into the code above.**

- **Step 1: initialize FLARE**.

First, we need to import NVIDIA FLARE, by simply doing
```python
import nvflare.client as flare
```
And before doing anything, it is important to perform necessary initialization by calling:
```python
flare.init()
```

Now let's look at the `main()` function above: this is where the computation logic happens. 

- **Step 2: receive model from server**.

The first code block in the original `main()` function above is getting the `input_model`:
```python
# Get a "mock" model: input_model
input_model = ...
```
However, in a federated paradigm, a client does not need to worry about how to get a model, it simply receives a copy of the global model from the server. With NVIDIA FLARE, this is done using the `receive()` API. Therefore we can replace this first code block by:
```python
input_model = flare.receive()
```
This tells the client that it will, at some point, receive a model from the server. As of when and how, that is the server's concern. As of the format of the `input_model` received from the server, NVIDIA FLARE uses a standard class [`flare.FLModel`](https://nvflare.readthedocs.io/en/main/programming_guide/fl_model.html#flmodel), which is a dictionary-like seriablizable class.

The rest of the code blocks in the `main()` function stays the same: we set default value to `input_model.params["numpy_key"]` if it's empty. Then we perform our simple `train()` operation (adding `1` to `input_model.params["numpy_key"]`) and `evaluate()` operation (compute mean of `input_model.params["numpy_key"]`).

- **Step 3: send new model to server**.

There is something missing in the end of the `main()` function though. Remember that in a federated paradigm, a client interacts with the server constantly, and needs to send the local computation results to the server, as soon as the local computation is finished. To do that, NVIDIA FLARE provides a convenient API `flare.send()`. As of the format of the results to be sent to the server, similar to `flare.receive()`, NVIDIA FLARE uses the standard `flare.FLModel` class. Adding the following in the end of the `main()` function, we send the computation results, a.k.a the modified `new_params` in our case, to the server:
```python
output_model = flare.FLModel(
            params={"numpy_key": new_params},
            params_type="FULL",
            metrics={"accuracy": metrics},
            current_round=input_model.current_round,
        )
flare.send(output_model)
```
Notice that appart from the `params` argument which refers to the parameters of the model / computation results, `flare.FLModel` has many other input arguments, such as `param_type`, `metrics`, `current_round`. Please refer to its [API documentation](https://nvflare.readthedocs.io/en/main/apidocs/nvflare.app_common.abstract.fl_model.html#module-nvflare.app_common.abstract.fl_model) for more detailed explanation of these arguments.

- **Step 4: rinse and repeat**.

At this point, we are almost finished on the client side, but there is still one last element missing. Remember that a federated workflow is usually an iterative interaction process between server and clients. The server decides on how many iterations, or "rounds" of computations to be performed by each client: this needs to be handled on the client side. With NVIDIA FLARE, this is done by wrapping the client computation inside a `while` clause:
```python
while flare.is_running():
    # client computation code
    ...
```
As the name indicates, the computation wrapped inside the `while flare.is_running():` will be performed for as many iterations as required by the server.

**And that's it: we have converted a centralized computation code to federated code in simply 4 steps!** We still need to create an Executor from this federated code, but you will see later that this is easy to do with some convenient APIs.

Putting all things together, we have the final federated client-side implementation:

```python
import copy
import numpy as np
import nvflare.client as flare # Import flare

def train(input_arr):
    output_arr = copy.deepcopy(input_arr)
    # mock training with plus 1
    return output_arr + 1

def evaluate(input_arr):
    # mock evaluation metrics
    return np.mean(input_arr)

def main():
   
    flare.init() # Initialization
    
    while flare.is_running(): # Run iteratively
        
        input_model = flare.receive() # Get copy of model from server

        if input_model.params == {}:
            params = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=np.float32)
        else:
            params = np.array(input_model.params["numpy_key"], dtype=np.float32)

        # training
        new_params = train(params)
        # evaluation
        metrics = evaluate(params)

        # Send results to server
        output_model = flare.FLModel(
            params={"numpy_key": new_params},
            params_type="FULL",
            metrics={"accuracy": metrics},
            current_round=input_model.current_round,
        )

        flare.send(output_model)

if __name__ == "__main__":
    main()
```

If you compare the federated implementation with the original one shown in the beginning of this section, you can notice that the difference is quite minimal: only additions of a few APIs. Though the computation is this example is quite trivial, this type of adaptation can be done on practically any complex computations, as we will explore in later content of this course. 

You can read the [documentation here](https://nvflare.readthedocs.io/en/main/programming_guide/execution_api_type/client_api.html) to learn more about the clien APIs.


### Server-Side Implementation

With the client-side implementation finished, let's now look at how to implement the server-side workflow, in file [examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py](examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py).

As we've explained above in FLARE's architecture, server-side implementation consists of creating Controllers, which defines server-side aggregation workflow and server-client interation logic. Compared with traditional centralized compute, the server-side code is completely new addition, therefore has to be implemented by users. Luckily, NVDIA FLARE provides an easy and extensible way to do it.

Below is the part of code that is related to server-side implementation:
```python
n_clients = 2
num_rounds = 3

persistor_id = job.to_server(NPModelPersistor(), "persistor")

# Define the controller workflow and send to server
controller = FedAvg(
    num_clients=n_clients,
    num_rounds=num_rounds,
    persistor_id=persistor_id,
)
```
In this example, the server-side workflow is quite straight-forward: computing the average from results received from clients. This corresponds exactly to the [FedAvg (Federated Averaging) Controller](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/fedavg.py#L18) that is already implemented in NVIDIA FLARE. This FedAvg Controller is instantiated with 3 arguments:
- `num_clients`: the number of clients to select for each training round. Notice that, this is different than the actual total number of participating clients. This parameter indicates the number of clients that will participate in each aggregation round. `num_clients` cannot be greater than the number of total clients, but it can be smaller, in which case, a random subset of `num_clients` clients will be selected for each round. Notice that, client sampling strategy is completely customizable in FLARE, as we will explain later on.
- `num_rounds`: total number of rounds of aggregation to be performed on the server-side.
- `persistor_id`: ID to a Persistor object.

A Persistor in NVIDIA FLARE is in charge of loading & serializing something, in this context, the global model on the server side. In this example, the global model is just a `numpy` array, therefore we use the class [`NPModelPersistor`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/np/np_model_persistor.py#L39) provided by NVIDIA FLARE. The `NPModelPersistor` implements:
- Default array initialization: this is typically for the initial round of a federated workflow, where the server needs to send a default global array to the client.
- `save_model()`: for serializing the array to the filesystem as a numpy `.npy` file.
- `load_model()`: for loading the array from the filesystem.

The ID of the created Persistor is passed to the `FedAvg` Controller, so that when the Controller needs to initialze the global model, saving the model to disk or loading a model from disk, it will call the corresponding function from the Persistor. FLARE provides many other default model Persistors, for instance the [`PTFileModelPersistor`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_opt/pt/file_model_persistor.py#L36) for `Pytorch` models, [`TFModelPersistor`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_opt/tf/model_persistor.py#L26) for Tensorflow models, etc. You also have the possibility to write your own model persistor by sub-classing the [ModelPersistor](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/abstract/model_persistor.py#L26) class. Also notice that, you do not have to use a Persistor object for model saving and loading: FLARE's base Controller class is flexible enough for you to implement your own model saving & loading logics, by simply overriding the corresponding functions. We will see this later on.

The `FedAvg` Controller is just an examplar reference Controller provided by FLARE. To better understand the fundamentals of how Controllers are implemented and can be customzied in FLARE, we need to look at the base class: the [ModelController](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/model_controller.py#L23) class. 

**The `ModelController` class is the fundational base Controller class, that allows developers to implement any custom server-side workflows.** This class has the following key APIs:
  - `run()` method: this is an asbtract function that must be overridden by all Controller classes that inherit from ModelController. This is the key function where the server-side workflow logic is actually implemented.
  - `load_model()` and `save_model()`: these are methods implementing how to load and save model on the server-side, including initializing the first model. In the `FedAvg` Controller, these methods actually call their corresponding function in the Persistor object that is passed to the `ModelController`. It is also perfectly ok to override these functions to accommodate your own customized model loading and saving, without relying on the concept of Persistors at all.
  - `aggregate()` and `update_model()`: these are core functions defining the global model aggregation and optimization algorithm on the server side
  - `sample_clients()`: this function queries the FLARE system and returns a list of participating clients for the current round. Noted that you can override this function to implement your own custom client sampling strategy.
  - `send_model()` and `send_model_and_wait()`: these functions implement communications between server and client via tasks.

The design of `ModelController` separate low-level communication codes from workflow-related codes, offering a flexible and easy way to custom any server-side workflow. In practice, developers typically do not need to worry about low-level communications that are handled by `send_model()` and `send_model_and_wait()`. All they need to re-implement is workflow-related logic:
  - the `run()` function which orchestrate the overall workflow
  - model loading & saving: via `load_model()` and `save_model()` functions
  - server-side model aggregation algorithm: via `aggregate()` and `update_model()`
  - clients sampling strategy: via `sample_clients()`
  - and any other custom routines needed by the workflow: you can implement any custom functions and plug them to the server workflow inside `run()` function 

To give a concrete example, let's look back at how Federated Averaging ([FedAvg](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/fedavg.py#L18) class & [BaseFedAvg](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/base_fedavg.py#L29) class) was implemented, inheritting from ModelController. The `run()` method is overriden in the [FedAvg](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/fedavg.py#L34) class:
 ```python
    def run(self) -> None:
        model = self.load_model()
        model.start_round = self.start_round
        model.total_rounds = self.num_rounds

        for self.current_round in range(self.start_round, self.start_round + self.num_rounds):
            model.current_round = self.current_round
            clients = self.sample_clients(self.num_clients)
            results = self.send_model_and_wait(targets=clients, data=model)
            aggregate_results = self.aggregate(results, aggregate_fn=self.aggregate_fn)
            model = self.update_model(model, aggregate_results)
            self.save_model(model)
```
Inside the `run()` function, a classic Federated Averaging workflow is orchestrated:
- An initial model is loaded with [`load_model()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/model_controller.py#L107) method, handled by the Persistor.
- For each round, we get a list of active clients via [`sample_clients()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/model_controller.py#L126), and send them each a copy of the current model for local training via [`send_model_and_wait()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/model_controller.py#L42). These functions contains low-level communication routines, already implemented by `ModelController`, so you can use these functions as is.
- When local clients return results for each round, we perform model aggregation via [`aggregate()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/base_fedavg.py#L115) function. The aggregation function is a custom function implemented in class [BaseFedAvg](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/base_fedavg.py#L115), and performs a simple weighted average of the clients local training results.
- When the aggregated result is ready, we update the global model via [`update_model()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/base_fedavg.py#L148) function. The model update function is a custom function implemented in class [BaseFedAvg](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/base_fedavg.py#L148).
- Finally, we save the model for each round, via the [`save_model()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/model_controller.py#L115) function.

As we can see, since `ModelController` already handles the low-level communications, and hide them from workflow-related logics, implementing server-side workflow using it allows developers to focus only on workflow-related implementation.

**And that's it: we've walked-through the implementation of server-side Controllers via leveraging the flexible ModelController class!**

In practice, the [FedAvg](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/fedavg.py) and [BaseFedAvg](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/base_fedavg.py) classes, built on top of `ModelController`, can be leveraged to implement most of the centralized federated workflows, with the flexibility to easily customize server-side aggregation function and model update function. NVIDIA FLARE also provides many other reference server-side workflow implementations, for instance, [cyclic workflow](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/cyclic.py), [Scaffold](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/scaffold.py), [cross-site evaluation](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/cross_site_eval.py) etc. 

You can read the [documentation here](https://nvflare.readthedocs.io/en/main/programming_guide/controllers/model_controller.html) to learn more about the `ModelController` with more examples.


### Putting Everything into a Federated Job

Now we have the client- & server-side implementation, all that remains is to put everything together into a Federated Job, so that it can be run by NVIDIA FLARE's runtime. This is done in file [examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py](examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py) (order of lines modified to facilitate explanation):
```python
from nvflare import FedJob
from nvflare.app_common.np.np_model_persistor import NPModelPersistor
from nvflare.app_common.widgets.intime_model_selector import IntimeModelSelector
from nvflare.app_common.workflows.fedavg import FedAvg
from nvflare.job_config.script_runner import FrameworkType, ScriptRunner

if __name__ == "__main__":

    job = FedJob(name="hello-fedavg-numpy")

    n_clients = 2
    num_rounds = 3
    
    persistor_id = job.to_server(NPModelPersistor(), "persistor")

    # Define the controller workflow and send to server
    controller = FedAvg(
        num_clients=n_clients,
        num_rounds=num_rounds,
        persistor_id=persistor_id,
    )
    job.to(controller, "server")

    job.to(IntimeModelSelector(key_metric="accuracy"), "server")

    # Add clients
    train_script = "src/hello-numpy_fl.py"
    for i in range(n_clients):
        executor = ScriptRunner(script=train_script, script_args="", framework=FrameworkType.NUMPY)
        job.to(executor, f"site-{i+1}")

    job.export_job("/tmp/nvflare/jobs/job_config")
    job.simulator_run("/tmp/nvflare/jobs/workdir", gpu="0")
```

Let's analyze this file. First, we define a `FedJob` object:
```python
    job = FedJob(name="hello-fedavg-numpy")
```
The [`FebJob`](https://nvflare.readthedocs.io/en/main/programming_guide/fed_job_api.html) class aims to model the federated workflow and server-client interactions, and it offers APIs to Pythonically define and configure federated jobs.

The next couple of lines are related to server-side Controller, as we've already seen in the server-side implementation section:
```python
    n_clients = 2
    num_rounds = 3
    
    persistor_id = job.to_server(NPModelPersistor(), "persistor")

    # Define the controller workflow and send to server
    controller = FedAvg(
        num_clients=n_clients,
        num_rounds=num_rounds,
        persistor_id=persistor_id,
    )
    job.to(controller, "server")
```
Here we set the number of participating clients for each round to be 2, and we set up for 3 rounds of "training". We also see the usage of `job.to` API: `job.to` sends an object to a specific target, where the target can be a client or the server. In FLARE, the runtime system requires that each instantiated component to be sent to its corresponding participants: any components that run on the server side, e.g., Controllers and Persistors, should be sent to server side, while any components that run on the client side, e.g., Executors, should be sent to the client side.

The API call: `job.to_server(...)` is equivalent to `job.to(..., "server")`. Therefore we can interpret the above usage as follows:
- `job.to_server(NPModelPersistor(), "persistor")`: sends an instance of `NPModelPersistor` to the server, give the instance an ID of "persistor"
- `job.to(controller, "server")`: send the FedAvg Controller to the server. This is equivalent to `job.to_server(controller)`

The next line:
```python
job.to(IntimeModelSelector(key_metric="accuracy"), "server")
```
sends another component, `IntimeModelSelector` to the server. We would not go into details of this component in this course, but all you need to know is that this component allows the server to select & save the best global model during federated training rounds, based on specific metric, which in this case is the "accuracy".  This is possible since each client sends an `FLModel` object to the server, which has an attribute `metrics` that the `IntimeModelSelector` can refer to. For more details, please check out the [documentation](https://nvflare.readthedocs.io/en/main/programming_guide/component_configuration.html#component-configuration-and-event-handling). In this example, the model selection based on a "mock" evaluation accuracy does not really make sense, but we will see its better use-case later in this course with other more realistic examples.

Next, we have a couple of lines that refer to the client-side code:
```python
    # Add clients
    train_script = "src/hello-numpy_fl.py"
    for i in range(n_clients):
        executor = ScriptRunner(script=train_script, script_args="", framework=FrameworkType.NUMPY)
        job.to(executor, f"site-{i+1}")
```
Here, we are essentially turning our client-side Python script [`examples/hello-fedavg-numpy/src/hello-numpy_fl.py`](examples/hello-fedavg-numpy/src/hello-numpy_fl.py) into an Executor object using the `ScriptRunner` API, and send it to each of the clients using `job.to` API. Most of the magic here is done by [`ScriptRunner`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/job_config/script_runner.py#L30), which alleviates the need to manually writing custom Executors. Instead, developers can start from a centralized training script, adapt it to a federated client-side local training script by adding FLARE Client APIs, and then convert it to an Executor using `ScriptRunner`.

Finally, the definition & configuration of a Federated Job is complete, we can export it to a local folder and run it later using different runtime backends:
```python
job.export_job("/tmp/nvflare/jobs/job_config")
```

**And we are all done: we have implemented and configured our first federated job using NVIDIA FLARE!**

In the next session, we will see how to run this federated job using FLARE's runtime backend.

### Running the Job with FL Simulator

As you probably saw, in the last line in file [examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py](examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py) , we have:
```python
job.simulator_run("/tmp/nvflare/jobs/workdir", gpu="0")
```

This line actually runs the federated job that we've just created directly with FL Simulator using `FedJob` API.

The [FL Simulator](https://nvflare.readthedocs.io/en/main/user_guide/nvflare_cli/fl_simulator.html) is a handy runtime designed for researchers to quickly test-run the federated job in a local development environment. It provides a convenient tool for fast prototyping and software / algorithm debugging before deploying in production. 

The first argument to [`job.simulator_run()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/job_config/api.py#L496) is a directory to run the federated job and save results. The function also accepts multiple other arguments, including total number of clients, number of threads, GPU index etc. See [here](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/job_config/api.py#L496) for details.

Noted that FL Simulator is designed for convenience, therefore it does not include necessary security features for real-world deployment. We will cover real-world deployment in later chapters of this course.

Let's run the job by executing the python script [examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py](examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py):


In [2]:
!python3 examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py

2024-10-09 18:52:42,228 - SimulatorRunner - INFO - Create the Simulator Server.
2024-10-09 18:52:42,230 - CoreCell - INFO - server: creating listener on tcp://0:57759
2024-10-09 18:52:42,240 - CoreCell - INFO - server: created backbone external listener for tcp://0:57759
2024-10-09 18:52:42,240 - ConnectorManager - INFO - 7730: Try start_listener Listener resources: {'secure': False, 'host': 'localhost'}
2024-10-09 18:52:42,241 - nvflare.fuel.f3.sfm.conn_manager - INFO - Connector [CH00002 PASSIVE tcp://0:11003] is starting
2024-10-09 18:52:42,742 - CoreCell - INFO - server: created backbone internal listener for tcp://localhost:11003
2024-10-09 18:52:42,742 - nvflare.fuel.f3.sfm.conn_manager - INFO - Connector [CH00001 PASSIVE tcp://0:57759] is starting
2024-10-09 18:52:42,785 - nvflare.fuel.hci.server.hci - INFO - Starting Admin Server localhost on Port 52871
2024-10-09 18:52:42,785 - SimulatorRunner - INFO - Deploy the Apps.
2024-10-09 18:52:42,789 - SimulatorRunner - INFO - Create 

There is quite a lot of console output, but you can essentially pick up a couple of prints, indicating that the initial `numpy` array `[[1,2,3], [4,5,6], [7,8,9]]` is incremented by 1 to each of its values after each round, and becoming `[[4,5,6], [7,8,9], [10,11,12]]` after a total of 3 rounds.


Let's also check the final best global model aggregated on the server-side. In this toy example, each client is simply adding 1 to the same fixed array for 3 rounds, so the average array aggregated at the server side should have the same value as the array returned from each client. Then, the server selects the best model based on the mock "accuracy", which is merely the mean value of the array. Therefore, we should expect the final best global model to contain the array with the largest mean value, i.e., `[[4,5,6], [7,8,9], [10,11,12]]`. Let's check if it is the case with the following code.

In [3]:
import numpy as np

server_model_path = "/tmp/nvflare/jobs/workdir/server/simulate_job/models/server.npy"
best_model = np.load(server_model_path)

print(best_model)

[[ 4.  5.  6.]
 [ 7.  8.  9.]
 [10. 11. 12.]]


We can see that, the final best global model indeed conatins the array with value `[[4,5,6], [7,8,9], [10,11,12]]`, as expected.

Alternatively you can also run the job using NVIDIA FLARE's CLI tool for FL Simulator: `nvflare simulator`. Let's have a look at the CLI's help page, by running the following command:

In [4]:
!nvflare simulator -h

usage: nvflare simulator [-h] [-w WORKSPACE] [-n N_CLIENTS] [-c CLIENTS]
                         [-t THREADS] [-gpu GPU] [-m MAX_CLIENTS]
                         [--end_run_for_all]
                         job_folder

positional arguments:
  job_folder

options:
  -h, --help            show this help message and exit
  -w WORKSPACE, --workspace WORKSPACE
                        WORKSPACE folder
  -n N_CLIENTS, --n_clients N_CLIENTS
                        number of clients
  -c CLIENTS, --clients CLIENTS
                        client names list
  -t THREADS, --threads THREADS
                        number of parallel running clients
  -gpu GPU, --gpu GPU   list of GPU Device Ids, comma separated
  -m MAX_CLIENTS, --max_clients MAX_CLIENTS
                        max number of clients
  --end_run_for_all     flag to indicate if running END_RUN event for all
                        clients


As you can, similar to [`job.simulator_run()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/job_config/api.py#L496), the CLI tool also provides similar options. We won't be changing much of the options here, so let's just proceed with the default values, apart from setting the workspace to `/tmp/nvflare/jobs/workdir-cli`. Feel free to experiment with different options of the CLI on your own.

Let's first export the job to a local folder. You probably need to uncomment the line:
```python
job.export_job("/tmp/nvflare/jobs/job_config")
```
and comment the line:
```python
job.simulator_run("/tmp/nvflare/jobs/workdir", gpu="0")
```
in [examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py](examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py)


Re-run the script to export the job under `/tmp/nvflare/jobs/workdir`:

In [5]:
!python3 examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py

Then execute the following command:

In [6]:
!nvflare simulator /tmp/nvflare/jobs/job_config/hello-fedavg-numpy -w /tmp/nvflare/jobs/workdir-cli

2024-10-09 18:55:05,151 - SimulatorRunner - WARNING - The number of threads is not provided. Set it to default: 1
2024-10-09 18:55:05,152 - SimulatorRunner - INFO - Create the Simulator Server.
2024-10-09 18:55:05,153 - CoreCell - INFO - server: creating listener on tcp://0:52091
2024-10-09 18:55:05,166 - CoreCell - INFO - server: created backbone external listener for tcp://0:52091
2024-10-09 18:55:05,166 - ConnectorManager - INFO - 8058: Try start_listener Listener resources: {'secure': False, 'host': 'localhost'}
2024-10-09 18:55:05,167 - nvflare.fuel.f3.sfm.conn_manager - INFO - Connector [CH00002 PASSIVE tcp://0:5203] is starting
2024-10-09 18:55:05,668 - CoreCell - INFO - server: created backbone internal listener for tcp://localhost:5203
2024-10-09 18:55:05,668 - nvflare.fuel.f3.sfm.conn_manager - INFO - Connector [CH00001 PASSIVE tcp://0:52091] is starting
2024-10-09 18:55:05,710 - nvflare.fuel.hci.server.hci - INFO - Starting Admin Server localhost on Port 55423
2024-10-09 18:

After the run is complete, we should expect exactly the same result:

In [7]:
np.load('/tmp/nvflare/jobs/workdir-cli/server/simulate_job/models/server.npy')

array([[ 4.,  5.,  6.],
       [ 7.,  8.,  9.],
       [10., 11., 12.]], dtype=float32)

As you can see, we get the same result as expected.

To finish up, let's remove temporary files generated by FLARE during the executions:

In [8]:
!rm -rf /tmp/nvflare/

Before continuing to the next chapter, let us do some exercises to make sure that you've grasped the essential of this chapter!

# Exercise 1

Change the client side "training" routine of the Hello FedAvg NumPy example: instead of adding value 1 to the parameter array, let's compute the square-root of each element of the array. Try exporting the new modified job, and run to inspect the results.

# Exercise 2

Change the server side workflow of the Hello FedAvg NumPy example: from Federated Averaging (FedAvg) to Cyclic Weight Transfer (CWT) workflow. In CWT workflow, for each round, the server passes model parameters from the previous client to the next one for iterative training, until it reaches the last client. Read [here](https://github.com/NVIDIA/NVFlare/tree/main/examples/hello-world/hello-cyclic) to learn more about CWT workflow: but be careful to not look into the code there immediately, it contains the solution for this exercise :p